# Base GLM-OCR vs Qwen 4B and 7B benchmark

Inference-only comparison on the exact 400-image validation split saved by `glm_ocr_from_base_curriculum.ipynb`. The three candidates are the untouched base GLM-OCR model, `Qwen/Qwen3-VL-4B-Instruct`, and `Qwen/Qwen2.5-VL-7B-Instruct`.

All three models are evaluated without competition fine-tuning. The Qwen models share one exact-transcription prompt; base GLM retains its native `Text Recognition:` prompt. All candidates receive the same source crop, the same external 384–1536 visual-token area policy, greedy decoding, and a 128-token generation limit. Qwen defaults to NF4 4-bit loading so the 7B model fits modest GPUs; the report records this setting.

Downloads are pinned to the Hugging Face commit resolved on first use. Predictions are cached atomically after every image, models are loaded one at a time, and reruns resume safely. This notebook does not train or change model selection. Run it from the project root.


In [ ]:
%pip install -r requirements-benchmark.txt


In [ ]:
from __future__ import annotations

import gc, hashlib, json, math, os, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from huggingface_hub import HfApi, snapshot_download
from IPython.display import display
from PIL import Image, ImageOps
from rapidfuzz.distance import Levenshtein
from tqdm.auto import tqdm
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
    GlmOcrForConditionalGeneration,
)

from src.transcription_metrics import TranscriptionMetrics, ZindiScore

ROOT = Path.cwd().resolve()
assert (ROOT / 'resources/Train.csv').is_file(), 'Start Jupyter in the project root.'
assert torch.cuda.is_available(), 'A CUDA GPU is required for this benchmark.'

RUN = ROOT / 'training_outputs/glm_ocr_from_base_curriculum_v7_native_ocr_balanced'
BASE_GLM = ROOT / 'model/glm-ocr'
IMAGES = ROOT / 'resources/images'
OUT = RUN / 'base_glm_qwen_4b_7b_benchmark'
OUT.mkdir(parents=True, exist_ok=True)

QWEN_SPECS = {
    'qwen3_vl_4b': {
        'repo_id': 'Qwen/Qwen3-VL-4B-Instruct',
        'local_dir': ROOT / 'model/qwen3-vl-4b-instruct',
    },
    'qwen2_5_vl_7b': {
        'repo_id': 'Qwen/Qwen2.5-VL-7B-Instruct',
        'local_dir': ROOT / 'model/qwen2.5-vl-7b-instruct',
    },
}
MODEL_ORDER = ('glm_ocr_base', 'qwen3_vl_4b', 'qwen2_5_vl_7b')
MODELS_TO_RUN = MODEL_ORDER  # Change to a subset to benchmark one model at a time.
QWEN_LOAD_IN_4BIT = True
MAX_NEW_TOKENS = 128
MIN_SPATIAL_PIXELS = 384 * 28 * 28
MAX_SPATIAL_PIXELS = 1536 * 28 * 28
GLM_PROMPT = 'Text Recognition:'
QWEN_PROMPT = ('Transcribe the handwritten text exactly. Preserve original spelling, capitalization, '
               'punctuation, abbreviations, and symbols. Output only the transcription.')
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0), '| dtype:', DTYPE, '| output:', OUT)


## Load the exact saved validation split

The benchmark does not recreate or resample validation. It reads the curriculum run's immutable validation manifest and checks it against the current `Train.csv`.


In [ ]:
run_identity = json.loads((RUN / 'run_identity.json').read_text(encoding='utf-8'))
run_config = run_identity['identity']['config']
assert run_config['holdout_size'] == 400 and run_config['seed'] == 42
assert run_config['prompt'] == GLM_PROMPT
assert run_config['min_spatial_pixels'] == MIN_SPATIAL_PIXELS
assert run_config['max_spatial_pixels'] == MAX_SPATIAL_PIXELS

manifest = pd.read_csv(RUN / 'validation_manifest.csv', dtype=str, keep_default_na=False)
validation = (manifest.loc[manifest.split.eq('validation')]
              .sort_values('ID').reset_index(drop=True))
source = pd.read_csv(ROOT / 'resources/Train.csv', dtype=str, keep_default_na=False)
source_targets = source.set_index('ID').Target
assert len(validation) == 400 and validation.ID.is_unique
assert validation.ID.isin(source_targets.index).all()
assert validation.Target.tolist() == source_targets.loc[validation.ID].tolist()
missing = [image_id for image_id in validation.ID if not (IMAGES / f'{image_id}.jpg').is_file()]
assert not missing, f'Missing validation images: {missing[:5]}'
print('Validation rows:', len(validation), '| GLM candidate: untouched base model')
display(validation[['label_group', 'height_band', 'length_band']].value_counts().head(12))


## Download and pin the two Qwen baselines

Each first download resolves `main` to an immutable commit SHA and writes it beside the local model. Later runs reuse that exact snapshot. Delete a Qwen model directory deliberately if you want to resolve a newer revision.


In [ ]:
def write_json_atomic(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

def ensure_qwen_snapshot(spec):
    local_dir = Path(spec['local_dir'])
    metadata_path = local_dir / 'benchmark_download.json'
    if metadata_path.is_file() and (local_dir / 'config.json').is_file():
        metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
        assert metadata['repo_id'] == spec['repo_id']
        return metadata
    revision = HfApi().model_info(spec['repo_id'], revision='main').sha
    snapshot_download(repo_id=spec['repo_id'], revision=revision, local_dir=local_dir)
    metadata = {'repo_id': spec['repo_id'], 'revision': revision}
    write_json_atomic(metadata_path, metadata)
    return metadata

qwen_revisions = {key: ensure_qwen_snapshot(spec) for key, spec in QWEN_SPECS.items()}
display(pd.DataFrame(qwen_revisions).T)


## Model-native encoding with controlled decoding

The external resize is identical for all candidates. GLM uses the exact native encoding and stop tokens from training; both Qwen models use the same Qwen transcription prompt and their official chat templates. No language-model fusion or higher-resolution routing is applied here, keeping this a direct neural-model comparison.


In [ ]:
def object_hash(value):
    payload = json.dumps(value, sort_keys=True, ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def normalize(value):
    return unicodedata.normalize('NFC', str(value)).strip()

def load_shared_image(image_id):
    with Image.open(IMAGES / f'{image_id}.jpg') as source_image:
        image = ImageOps.exif_transpose(source_image).convert('RGB')
    for bound, comparison in [(MAX_SPATIAL_PIXELS, lambda area, limit: area > limit),
                              (MIN_SPATIAL_PIXELS, lambda area, limit: area < limit)]:
        if comparison(image.width * image.height, bound):
            scale = math.sqrt(bound / (image.width * image.height))
            image = image.resize(
                (max(1, int(image.width * scale)), max(1, int(image.height * scale))),
                resample=Image.Resampling.BICUBIC,
            )
    return image

def load_glm_runtime():
    processor = AutoProcessor.from_pretrained(BASE_GLM, local_files_only=True, use_fast=False)
    base = GlmOcrForConditionalGeneration.from_pretrained(
        BASE_GLM, local_files_only=True, dtype=DTYPE, device_map={'': 0},
        attn_implementation='sdpa',
    )
    model = base.eval()
    generation = json.loads((BASE_GLM / 'generation_config.json').read_text(encoding='utf-8'))
    eos_ids = generation['eos_token_id']
    eos_ids = [eos_ids] if isinstance(eos_ids, int) else list(eos_ids)
    eos_ids = sorted(set(eos_ids + [processor.tokenizer.eos_token_id]))
    return {'model': model, 'base': base, 'processor': processor, 'eos_ids': eos_ids}

def load_qwen_runtime(key):
    local_dir = Path(QWEN_SPECS[key]['local_dir'])
    processor = AutoProcessor.from_pretrained(local_dir, local_files_only=True, trust_remote_code=True)
    load_options = {
        'local_files_only': True, 'trust_remote_code': True, 'device_map': {'': 0},
        'dtype': DTYPE, 'attn_implementation': 'sdpa',
    }
    if QWEN_LOAD_IN_4BIT:
        load_options['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=DTYPE,
        )
    model = AutoModelForImageTextToText.from_pretrained(local_dir, **load_options).eval()
    return {'model': model, 'processor': processor}

def predict_glm(runtime, image):
    processor, model = runtime['processor'], runtime['model']
    prompt = ('[gMASK]<sop><|user|>\n<|begin_of_image|><|image|><|end_of_image|>'
              + GLM_PROMPT + '<|assistant|>')
    inputs = processor(text=[prompt], images=[image], add_special_tokens=False, return_tensors='pt')
    inputs.pop('token_type_ids', None)
    prompt_length = int(inputs['input_ids'].shape[1])
    inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False, use_cache=True,
            eos_token_id=runtime['eos_ids'], pad_token_id=processor.tokenizer.pad_token_id,
        )
    generated = output[0, prompt_length:].tolist()
    prediction = normalize(processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    hit_limit = len(generated) >= MAX_NEW_TOKENS and generated[-1] not in runtime['eos_ids']
    return prediction, len(generated), bool(hit_limit)

def predict_qwen(runtime, image):
    processor, model = runtime['processor'], runtime['model']
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': QWEN_PROMPT},
    ]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    )
    inputs.pop('token_type_ids', None)
    prompt_length = int(inputs['input_ids'].shape[1])
    inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, num_beams=1, do_sample=False, use_cache=True,
        )
    generated = output[0, prompt_length:].tolist()
    prediction = normalize(processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    eos_ids = model.generation_config.eos_token_id
    eos_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or [])
    hit_limit = len(generated) >= MAX_NEW_TOKENS and (not generated or generated[-1] not in eos_ids)
    return prediction, len(generated), bool(hit_limit)

model_signatures = {
    'glm_ocr_base': object_hash({
        'base_files': run_identity['identity']['base_files'], 'prompt': GLM_PROMPT,
        'min_pixels': MIN_SPATIAL_PIXELS, 'max_pixels': MAX_SPATIAL_PIXELS,
        'max_new_tokens': MAX_NEW_TOKENS,
    }),
    **{key: object_hash({
        **qwen_revisions[key], 'prompt': QWEN_PROMPT, 'four_bit': QWEN_LOAD_IN_4BIT,
        'min_pixels': MIN_SPATIAL_PIXELS, 'max_pixels': MAX_SPATIAL_PIXELS,
        'max_new_tokens': MAX_NEW_TOKENS,
    }) for key in QWEN_SPECS},
}


## Run sequentially with per-image resume

The cell loads only one model at a time. To run across several sessions, set `MODELS_TO_RUN` to one model key, finish it, restart the kernel if desired, and run the next key. Existing compatible cache entries are reused.


In [ ]:
assert set(MODELS_TO_RUN).issubset(MODEL_ORDER)

def cache_path(model_key, image_id):
    return OUT / 'cache' / model_key / f'{image_id}.json'

def read_compatible_cache(model_key, image_id):
    path = cache_path(model_key, image_id)
    if not path.is_file():
        return None
    value = json.loads(path.read_text(encoding='utf-8'))
    return value if value.get('signature') == model_signatures[model_key] else None

for model_key in MODELS_TO_RUN:
    pending = [image_id for image_id in validation.ID
               if read_compatible_cache(model_key, image_id) is None]
    print(f'{model_key}: {len(validation) - len(pending)}/{len(validation)} cached')
    if not pending:
        continue
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    runtime = load_glm_runtime() if model_key == 'glm_ocr_base' else load_qwen_runtime(model_key)
    model = runtime['model']
    parameters = int(sum(parameter.numel() for parameter in model.parameters()))
    started = time.perf_counter()
    try:
        for image_id in tqdm(pending, desc=model_key):
            image = load_shared_image(image_id)
            torch.cuda.synchronize(); item_started = time.perf_counter()
            if model_key == 'glm_ocr_base':
                prediction, generated_tokens, hit_limit = predict_glm(runtime, image)
            else:
                prediction, generated_tokens, hit_limit = predict_qwen(runtime, image)
            torch.cuda.synchronize(); seconds = time.perf_counter() - item_started
            write_json_atomic(cache_path(model_key, image_id), {
                'signature': model_signatures[model_key], 'ID': image_id,
                'Prediction': prediction, 'generated_tokens': generated_tokens,
                'hit_token_limit': hit_limit, 'seconds': seconds,
            })
        runtime_metadata = {
            'model': model_key, 'signature': model_signatures[model_key],
            'parameters': parameters, 'new_images': len(pending),
            'wall_seconds': time.perf_counter() - started,
            'peak_cuda_gib': torch.cuda.max_memory_allocated() / 1024**3,
            'qwen_load_in_4bit': QWEN_LOAD_IN_4BIT if model_key != 'glm_ocr_base' else None,
        }
        write_json_atomic(OUT / f'{model_key}_runtime.json', runtime_metadata)
    finally:
        del model, runtime
        gc.collect(); torch.cuda.empty_cache()


## Score and compare

The main table uses the same weighted WER/CER proxy as training. Cohort tables expose whether aggregate differences come from ordinary, complex, suspect, short, or tall crops. Pairwise outcomes count lower per-image character distance as an improvement.


In [ ]:
metric = TranscriptionMetrics(score=ZindiScore(
    word_normalizer=run_config['word_normalizer'],
    character_normalizer=run_config['character_normalizer'],
))
frames, summaries = {}, []
for model_key in MODEL_ORDER:
    rows = [read_compatible_cache(model_key, image_id) for image_id in validation.ID]
    missing_count = sum(row is None for row in rows)
    if missing_count:
        print(f'Skipping {model_key}: {missing_count} predictions are missing.')
        continue
    predictions = pd.DataFrame(rows)
    frame = validation.merge(predictions, on='ID', validate='one_to_one')
    scores = metric(frame.Target, frame.Prediction)
    summaries.append({
        'model': model_key, 'rows': len(frame), **scores,
        'exact_match_rate': float(frame.Target.map(normalize).eq(frame.Prediction).mean()),
        'blank_rate': float(frame.Prediction.eq('').mean()),
        'token_limit_rate': float(frame.hit_token_limit.mean()),
        'mean_seconds_per_image': float(frame.seconds.mean()),
    })
    frames[model_key] = frame
    frame.to_csv(OUT / f'{model_key}_predictions.csv', index=False, encoding='utf-8')

summary = pd.DataFrame(summaries).sort_values('zindi_score_proxy', ascending=False).reset_index(drop=True)
summary.to_csv(OUT / 'benchmark_summary.csv', index=False)
display(summary)

cohort_rows = []
for model_key, frame in frames.items():
    for column in ('label_group', 'height_band', 'length_band'):
        for cohort, group in frame.groupby(column, observed=True):
            cohort_rows.append({
                'model': model_key, 'cohort_type': column, 'cohort': cohort, 'rows': len(group),
                **metric(group.Target, group.Prediction),
            })
cohorts = pd.DataFrame(cohort_rows)
cohorts.to_csv(OUT / 'benchmark_cohorts.csv', index=False)
display(cohorts.sort_values(['cohort_type', 'cohort', 'zindi_score_proxy'],
                            ascending=[True, True, False]))

if 'glm_ocr_base' in frames:
    reference = frames['glm_ocr_base'].set_index('ID')
    pairwise = []
    for model_key, frame in frames.items():
        if model_key == 'glm_ocr_base':
            continue
        candidate = frame.set_index('ID').loc[reference.index]
        glm_errors = np.array([Levenshtein.distance(t, p) for t, p in
                               zip(reference.Target.map(normalize), reference.Prediction)])
        qwen_errors = np.array([Levenshtein.distance(t, p) for t, p in
                                zip(candidate.Target.map(normalize), candidate.Prediction)])
        qwen_score = summary.set_index('model').loc[model_key, 'zindi_score_proxy']
        glm_score = summary.set_index('model').loc['glm_ocr_base', 'zindi_score_proxy']
        pairwise.append({
            'candidate': model_key, 'reference': 'glm_ocr_base',
            'score_delta_candidate_minus_glm': float(qwen_score - glm_score),
            'candidate_better_rows': int((qwen_errors < glm_errors).sum()),
            'tied_rows': int((qwen_errors == glm_errors).sum()),
            'candidate_worse_rows': int((qwen_errors > glm_errors).sum()),
        })
    pairwise = pd.DataFrame(pairwise)
    pairwise.to_csv(OUT / 'benchmark_pairwise_vs_glm.csv', index=False)
    display(pairwise)

report = {
    'validation_rows': len(validation), 'validation_run_key': run_identity['run_key'],
    'glm_candidate': 'base', 'glm_base_files': run_identity['identity']['base_files'],
    'qwen_revisions': qwen_revisions,
    'qwen_load_in_4bit': QWEN_LOAD_IN_4BIT,
    'max_new_tokens': MAX_NEW_TOKENS, 'min_spatial_pixels': MIN_SPATIAL_PIXELS,
    'max_spatial_pixels': MAX_SPATIAL_PIXELS,
    'models_completed': list(frames),
    'summary': summary.to_dict(orient='records'),
}
write_json_atomic(OUT / 'benchmark_report.json', report)
print('Saved benchmark reports to:', OUT)
